# Setup

In [ ]:
import os
from getpass import getpass

# This will pop up an input box at the top of VS Code when you run the cell
os.environ["GROQ_API_KEY"] = getpass("Enter your GROQ_API_KEY: ")

print("Key loaded successfully!")

In [ ]:
%pip install langchain langchain-groq langchain-community langchain-text-splitters pymupdf pydantic rich arxiv

# Working with documents

A **Document Loader** is a class that:

1. Fetches content from some source (Arxiv, a local PDF, a website, …)

2. Returns a Python list of LangChain `Document` objects

Every loader returns `list[Document]`. Each `Document` contains:

1. `document.page_content`

2. `document.meta_data` 

### Arxiv - Load Research papers

In [ ]:
import arxiv
import urllib.request
import fitz
from langchain_core.documents import Document

def load_arxiv_papers(query:str, load_max_docs: int = 1) -> list[Document]:
    client = arxiv.Client()
    search = arxiv.Search(query=query, max_results=load_max_docs)
    documents = []

    for result in client.results(search):
        # Download the PDF to a temp file
        pdf_path = f"/tmp/{result.entry_id.split('/')[-1]}.pdf"
        urllib.request.urlretrieve(result.pdf_url, pdf_path)

        # Extract full text with PyMuPDF
        full_text = ""
        with fitz.open(pdf_path) as pdf:
            for page in pdf:
                full_text += page.get_text()

        documents.append(Document(
            page_content=full_text,
            metadata={
                "title":     result.title,
                "authors":   ", ".join(str(a) for a in result.authors),
                "published": str(result.published.date()),
                "summary":   result.summary,
                "entry_id":  result.entry_id,
                "pdf_url":   result.pdf_url,
            }
        ))

    return documents

documents = load_arxiv_papers(query="RAG", load_max_docs=2)
print(documents[0].page_content, "\n")

### Local File Loaders

In [ ]:
%pip install "unstructured[pdf]" --break-system-packages

#### UnstructuredFileLoader
How it parses: 

It attempts to reconstruct the structural layout of the file. It partitions the document into individual semantic blocks (e.g. distinguishing a "Title" from a "Paragraph").

Modes:

1. mode="single" (default): Combines all extracted text into one large Document.

2. mode="elements": Returns a list of separate Document objects, where each object is a specific element (like a separate table, title, list item, or paragraph).

In [ ]:
from ipywidgets import FileUpload
from IPython.display import display

upload = FileUpload(accept='', multiple=False)
display(upload)

In [ ]:
filename = list(upload.value.keys())[0]
with open(filename, "wb") as f:
    f.write(upload.value[filename]["content"])

# you can pass file name to any loader
print(upload)

In [ ]:
from langchain_community.document_loaders import PyMuPDFLoader

loader = PyMuPDFLoader("resume.pdf")
docs = loader.load()

print(len(docs))
print(docs[0].page_content[:500])

## Chunking

A typical research paper has 50,000 - 150,000 Characters, far too long to fit in a single LLM Call. We split it into smaller chunks that fit comfortably in context window

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=600, # Max char per chunk
    chunk_overlap=100, # How many char next chunk uses from previous for overlap.
    separators=["\n\n", "\n", ".", ";", ",", " ", ""], # Separators to try in order
    length_function=len
)

splits = text_splitter.split_documents(docs)

In [ ]:
print(f"Number Of Chunks: {len(splits)}")
print(f"Avg Chunk Lenght: {sum(len(d.page_content) for d in splits) // len(splits)} chars")
for i in [0, 1, 2, -1]:
    print(f"\n[Chunk {i}] ({len(splits[i].page_content)} chars)")
    print(splits[i].page_content)
    print("─" * 60)

Token based chunking:

The split above counts characters. But LLM's have limits measured in tokens(roughly 1 token = 0.75 words). For more accurate splitting

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    encoding_name="cl100k_base",
    chunk_size=300,
    chunk_overlap=30,
)

In [ ]:
splits = text_splitter.split_documents(docs)
print(f"Number Of Chunks: {len(splits)}")
print(f"Avg Chunk Lenght: {sum(len(d.page_content) for d in splits) // len(splits)} chars")
for i in [0, 1, 2, -1]:
    print(f"\n[Chunk {i}] ({len(splits[i].page_content)} chars)")
    print(splits[i].page_content)
    print("─" * 60)

### Pydantic Models

We want LLMs to return structured data, pydantic models make it easy for us

In [ ]:
from pydantic import BaseModel, Field
from typing import List

class DocumentSummaryBase(BaseModel):
    running_summary: str = Field(
        "",  # default value
        
        # description (llm read this)
        description="Running description of the document. Do not override; only update!"
    )
    main_ideas: List[str] = Field(
        [],
        description="Most important information from the document (max 3)"
    )
    loose_ends: List[str] = Field(
        [],
        description="Open questions to incorporate into future summaries (max 3)"
    )

In [ ]:
from langchain_core.output_parsers import PydanticOutputParser

parser = PydanticOutputParser(pydantic_object=DocumentSummaryBase)

instructions = parser.get_format_instructions()